In [1]:
import requests

import pandas as pd

def fetch_forecast(latitude=51.5 , longitude=-0.1, forecast_days=16):

#forecast days can be 1, 3, 7, 14, 16

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ["temperature_2m",
                   "wind_gusts_10m",
                   "cloud_cover",
                   "direct_radiation",
                   "diffuse_radiation",
                   "shortwave_radiation",
                   "wind_speed_120m",
                   "wind_speed_80m",
                   "pressure_msl",
                   "precipitation"],
        "timezone": "GMT",
        "forecast_days": forecast_days,
        "wind_speed_unit": "ms",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"Unexpected API response:{data}")

    df = pd.DataFrame(data["hourly"])

    return df

In [7]:
def weather_preproc_no_fill(df):
    ''' preprocess weather dataframe, resample, rename, check quality'''
    # datetime and set index
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

    df['wind_speed_100m'] =  (df['wind_speed_120m'] + df['wind_speed_80m'])/2

    # rename columns w/ units
    df = df.rename(columns={
        'temperature_2m': 'temperature_2m_c',
        'wind_speed_100m': 'wind_speed_100m_ms',
        'wind_gusts_10m': 'wind_gusts_10m_ms',
        'cloud_cover': 'cloud_cover_pct',
        'shortwave_radiation': 'shortwave_radiation_wm2',
        'direct_radiation': 'direct_radiation_wm2',
        'diffuse_radiation': 'diffuse_radiation_wm2',
        'pressure_msl': 'pressure_msl_hpa',
        'precipitation': 'precipitation_mm'
    })

    return df

In [8]:
df = fetch_forecast()
df = weather_preproc_no_fill(df)

In [9]:
df

,temperature_2m_c,wind_gusts_10m_ms,cloud_cover_pct,direct_radiation_wm2,diffuse_radiation_wm2,shortwave_radiation_wm2,wind_speed_120m,wind_speed_80m,pressure_msl_hpa,precipitation_mm,wind_speed_100m_ms
time,,,,,,,,,,,
2026-03-13 00:00:00,11.4,17.5,100,0.0,0.0,0.0,13.67,14.62,999.6,0.0,14.145
2026-03-13 01:00:00,11.4,17.7,100,0.0,0.0,0.0,13.86,15.36,998.3,0.3,14.610
2026-03-13 02:00:00,11.5,17.5,100,0.0,0.0,0.0,13.91,15.81,997.3,1.7,14.860
2026-03-13 03:00:00,10.8,14.6,100,0.0,0.0,0.0,11.72,12.43,998.3,2.5,12.075
2026-03-13 04:00:00,7.9,10.1,100,0.0,0.0,0.0,9.59,11.20,999.4,0.6,10.395
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-28 19:00:00,10.2,6.6,100,4.0,4.0,8.0,6.46,6.04,1026.1,0.0,6.250
2026-03-28 20:00:00,9.7,6.5,100,0.0,0.0,0.0,5.92,5.52,1026.0,0.0,5.720
2026-03-28 21:00:00,9.2,6.6,100,0.0,0.0,0.0,5.48,5.10,1025.7,0.0,5.290
